# Project 2 — Fraud Detection in Financial Transactions

**Objective:** Analyze transaction data to identify patterns of fraudulent behavior, build a Machine Learning model to detect fraud, and improve accuracy using feature engineering and ML algorithms.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE

## Step 1: Load Dataset

In [ ]:
df = pd.read_csv('2_fraud_detection_sample.csv')
print(f'Shape: {df.shape}')
df.head()

## Step 2: EDA — Class Distribution

In [ ]:
print('Class Distribution:')
print(df['Class'].value_counts())
print(f'Fraud rate: {df["Class"].mean()*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

class_counts = df['Class'].value_counts()
axes[0].bar(['Non-Fraud (0)', 'Fraud (1)'], class_counts.values,
            color=['steelblue', 'crimson'], edgecolor='black')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

df.groupby('Class')['Amount'].plot(kind='hist', bins=30, alpha=0.7, ax=axes[1], legend=True)
axes[1].set_title('Transaction Amount by Class')
axes[1].set_xlabel('Amount')
axes[1].legend(['Non-Fraud', 'Fraud'])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), cmap='coolwarm', center=0, linewidths=0.3)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## Step 3: Feature Engineering

In [ ]:
df['Amount_log']       = np.log1p(df['Amount'])
df['Amount_zscore']    = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
df['High_Amount_Flag'] = (df['Amount'] > df['Amount'].quantile(0.95)).astype(int)
print('Features after engineering:', df.columns.tolist())

## Step 4: Split Features & Target

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## Step 5: Handle Class Imbalance with SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print('After SMOTE:')
print(pd.Series(y_train_res).value_counts())

## Step 6: Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

## Step 7: Train Random Forest Classifier

In [ ]:
model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train_res)
print('✅ Model Trained')

## Step 8: Evaluate Model

In [ ]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print('Confusion Matrix:')
cm = confusion_matrix(y_test, y_pred)
print(cm)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Non-Fraud', 'Fraud']))
print(f'ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}')

## Step 9: Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Fraud', 'Fraud'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

roc_auc = roc_auc_score(y_test, y_prob)
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
axes[1].plot([0,1],[0,1],'--',color='navy')
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

feat_imp = pd.Series(model.feature_importances_, index=X.columns)
feat_imp.nlargest(10).plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Top 10 Feature Importances')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

## Conclusion
- Dataset was highly imbalanced (96% non-fraud, 4% fraud). SMOTE balanced the training set.
- Feature engineering (log transform, z-score, high-amount flag) improved signal.
- Random Forest with `class_weight='balanced'` was trained and evaluated.
- ROC AUC, confusion matrix, and classification report all computed.
- Feature importances reveal which transaction features drive fraud detection.